In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Project 2: Exploratory Data Analysis (EDA)\n",
        "## Floral Extract Yield & Efficacy Predictor\n",
        "\n",
        "This notebook explores `data/tabular/floral_extract_data.csv` to inspect data quality, distributions, feature relationships, and correlation with `yield_pct` before training regression models (`RandomForestRegressor` / `XGBoost`)."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "import matplotlib.pyplot as plt\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "import seaborn as sns\n",
        "\n",
        "sns.set_theme(style=\"whitegrid\")\n",
        "\n",
        "# Resolve paths whether notebook is run from repo root or project2_regressor/\n",
        "cwd = Path.cwd().resolve()\n",
        "project_root = cwd.parent if cwd.name == \"project2_regressor\" else cwd\n",
        "data_path = project_root / \"data\" / \"tabular\" / \"floral_extract_data.csv\"\n",
        "assets_dir = project_root / \"assets\"\n",
        "assets_dir.mkdir(parents=True, exist_ok=True)\n",
        "\n",
        "df = pd.read_csv(data_path)\n",
        "print(f\"Loaded dataset from: {data_path}\")\n",
        "print(f\"Shape: {df.shape[0]} rows x {df.shape[1]} columns\")\n",
        "df.head()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1. Data Types, Missing Values & Summary Statistics"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(\"--- Info & Missing Values ---\")\n",
        "df.info()\n",
        "print(\"\\nMissing values per column:\")\n",
        "print(df.isna().sum())\n",
        "print(f\"Duplicate rows: {df.duplicated().sum()}\")\n",
        "\n",
        "print(\"\\n--- Summary Statistics ---\")\n",
        "df.describe().T.round(2)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. Feature & Target Distributions"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()\n",
        "fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n",
        "axes = axes.flatten()\n",
        "\n",
        "for idx, col in enumerate(numeric_cols):\n",
        "    sns.histplot(df[col], kde=True, bins=25, ax=axes[idx], color=\"#2b7a78\")\n",
        "    axes[idx].set_title(f\"Distribution of {col}\")\n",
        "\n",
        "# Hide extra subplot(s) if fewer than 6 numeric columns\n",
        "for idx in range(len(numeric_cols), len(axes)):\n",
        "    axes[idx].axis(\"off\")\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. Laboratory Parameters vs. Extraction Yield (`yield_pct`)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "feature_cols = [c for c in numeric_cols if c != \"yield_pct\"]\n",
        "n_feats = len(feature_cols)\n",
        "cols_grid = 2\n",
        "rows_grid = int(np.ceil(n_feats / cols_grid))\n",
        "\n",
        "fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(14, 5 * rows_grid))\n",
        "axes = np.atleast_1d(axes).flatten()\n",
        "\n",
        "for idx, feat in enumerate(feature_cols):\n",
        "    sns.scatterplot(data=df, x=feat, y=\"yield_pct\", alpha=0.45, s=25, ax=axes[idx], color=\"#17252a\")\n",
        "    # Bin mean trendline to highlight non-linear behavior\n",
        "    binned = df.groupby(pd.cut(df[feat], bins=12), observed=False)[\"yield_pct\"].mean()\n",
        "    bin_centers = [interval.mid for interval in binned.index]\n",
        "    axes[idx].plot(bin_centers, binned.values, color=\"#d95f02\", linewidth=2.5, marker=\"o\", label=\"Binned Mean\")\n",
        "    axes[idx].set_title(f\"{feat} vs. yield_pct\")\n",
        "    axes[idx].legend()\n",
        "\n",
        "for idx in range(n_feats, len(axes)):\n",
        "    axes[idx].axis(\"off\")\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4. Correlation Matrix & Heatmap"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "corr_matrix = df[numeric_cols].corr()\n",
        "print(\"Correlation with yield_pct:\")\n",
        "print(corr_matrix[\"yield_pct\"].sort_values(ascending=False).round(4))\n",
        "\n",
        "fig, ax = plt.subplots(figsize=(8, 6))\n",
        "sns.heatmap(\n",
        "    corr_matrix,\n",
        "    annot=True,\n",
        "    fmt=\".2f\",\n",
        "    cmap=\"coolwarm\",\n",
        "    center=0,\n",
        "    square=True,\n",
        "    linewidths=0.5,\n",
        "    ax=ax,\n",
        ")\n",
        "ax.set_title(\"Feature & Target Correlation Heatmap\")\n",
        "plt.tight_layout()\n",
        "\n",
        "heatmap_path = assets_dir / \"correlation_heatmap.png\"\n",
        "fig.savefig(heatmap_path, dpi=200)\n",
        "print(f\"Saved heatmap to: {heatmap_path}\")\n",
        "plt.show()"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.10.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}